# Notebook Fusion — Pipeline complet

Ce notebook regroupe l'ensemble du travail :
1. **Transformation des donnees** (KNN imputation, nettoyage outliers, normalisation Z-Score)
2. **Regression lineaire** (statsmodels OLS)
3. **Random Forest**
4. **Perceptron Multi-Couches (MLP)**
5. **Industrialisation** (predict.py)

---
# PARTIE 1 — Pipeline de transformation des donnees

**Source** : `housing.csv` (donnees brutes) → **Sortie** : `amputed_normalized_housing_data.csv`

| Etape | Operation | Detail |
|-------|-----------|--------|
| 1 | Imputation KNN | `total_bedrooms` — 207 NaN combles par $k=5$ voisins ponderes par distance |
| 2 | Encodage ordinal | `ocean_proximity` → 0–4 |
| 3 | Nettoyage outliers | Suppression prix plafonnes (500 001$) + IQR sur features numeriques |
| 4 | Normalisation Z-Score | $z = \frac{x - \mu}{\sigma}$ sur toutes les features sauf `median_house_value` (cible) |

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

housing = pd.read_csv('./datasets/housing.csv')
print(f"Shape : {housing.shape}")
print(f"Valeurs manquantes : total_bedrooms → {housing['total_bedrooms'].isnull().sum()}")

## 1. Imputation KNN ($k=5$, ponderation par distance)

$$\hat{x}_i = \frac{\sum_{j \in \mathcal{N}_k(i)} w_j \cdot x_j}{\sum_{j \in \mathcal{N}_k(i)} w_j}, \quad w_j = \frac{1}{d(i,j)}$$

On standardise temporairement avant KNN pour que la distance euclidienne soit coherente entre colonnes.

In [ ]:
# Encodage temporaire de ocean_proximity pour KNN (numérique requis)
ocean_mapping = {'<1H OCEAN': 0, 'INLAND': 1, 'ISLAND': 2, 'NEAR BAY': 3, 'NEAR OCEAN': 4}
housing['ocean_proximity'] = housing['ocean_proximity'].map(ocean_mapping)

# Standardisation temporaire → KNN → dé-standardisation
num_cols = housing.columns.tolist()
means, stds = housing[num_cols].mean(), housing[num_cols].std()
X_scaled = (housing[num_cols] - means) / stds

imputer = KNNImputer(n_neighbors=5, weights='distance')
X_imputed = pd.DataFrame(
    imputer.fit_transform(X_scaled) * stds.values + means.values,
    columns=num_cols, index=housing.index
)
housing['total_bedrooms'] = X_imputed['total_bedrooms'].round(0)

print(f"Valeurs manquantes après imputation : {housing.isnull().sum().sum()}")

## 2. Nettoyage des outliers

**2a.** Suppression des prix plafonnes (`median_house_value == 500001`) — donnees censurees, pas des prix reels.

**2b.** Suppression des outliers IQR sur les features numeriques (sauf `ocean_proximity` et `median_house_value`) :

$$\text{outlier si } x < Q_1 - 1.5 \times IQR \text{ ou } x > Q_3 + 1.5 \times IQR$$

In [ ]:
n_before = len(housing)

# 2a. Suppression des prix plafonnés
capped = (housing['median_house_value'] == 500001).sum()
housing = housing[housing['median_house_value'] != 500001]
print(f"Prix plafonnés supprimés : {capped}")

# 2b. Suppression des outliers IQR sur les features numériques
iqr_cols = [c for c in housing.columns if c not in ('ocean_proximity', 'median_house_value')]
mask = pd.Series(True, index=housing.index)

for col in iqr_cols:
    q1 = housing[col].quantile(0.25)
    q3 = housing[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    col_mask = (housing[col] >= lower) & (housing[col] <= upper)
    n_out = (~col_mask).sum()
    if n_out > 0:
        print(f"  {col:>20s} : {n_out} outliers (bornes [{lower:.1f}, {upper:.1f}])")
    mask &= col_mask

housing = housing[mask].reset_index(drop=True)
n_after = len(housing)
print(f"\nLignes supprimées : {n_before - n_after} ({(n_before - n_after)/n_before*100:.1f}%)")
print(f"Lignes restantes  : {n_after}")

## 3. Normalisation Z-Score (features uniquement)

$$z = \frac{x - \mu}{\sigma}$$

`median_house_value` reste en valeur brute — c'est la cible du perceptron.

In [ ]:
target = 'median_house_value'
feature_cols = [c for c in housing.columns if c != target]

scaler = StandardScaler()
housing[feature_cols] = scaler.fit_transform(housing[feature_cols])

print("Features (mean ≈ 0, std ≈ 1) :")
print(housing[feature_cols].describe().loc[['mean', 'std']].round(4))
print(f"\nmedian_house_value (non transformé) : mean={housing[target].mean():.0f}, std={housing[target].std():.0f}")

## 4. Export

In [ ]:
output_path = './datasets/amputed_normalized_housing_data.csv'
housing.to_csv(output_path, index=False)
print(f"Exporté : {output_path} — {housing.shape[0]} lignes, {housing.shape[1]} colonnes")
housing.head()

In [ ]:
import joblib

# Sauvegarder le scaler pour réutilisation dans predict.py
joblib.dump(scaler, './models/scaler.joblib')
joblib.dump(feature_cols, './models/feature_cols.joblib')
print("Scaler exporté → ./models/scaler.joblib")
print(f"Features scalées : {feature_cols}")

---
# PARTIE 2 — Regression Lineaire (OLS)

**Modele** : Regression lineaire (OLS — Ordinary Least Squares)

$$\hat{y} = X \theta \quad \text{avec} \quad \theta = (X^T X)^{-1} X^T y$$

**Pipeline** :
1. Chargement des donnees nettoyees (imputees, sans outliers, normalisees Z-Score)
2. Analyse de colinearite et selection des features
3. Train/test split (70/30)
4. Entrainement du modele de regression lineaire
5. Evaluation de la precision via la **moyenne** et l'**ecart-type** des erreurs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

df = pd.read_csv('./datasets/amputed_normalized_housing_data.csv')
print(f"Shape : {df.shape}")
df.head()

## Observation de la linearite — Feature vs Prix median

Avant de construire un modele lineaire, on visualise la relation entre chaque feature numerique et la cible (`median_house_value`).  
Une **droite de regression** est superposee pour evaluer visuellement si la relation est lineaire. Le coefficient de correlation $r$ est affiche pour chaque graphique.

In [ ]:
target = 'median_house_value'
num_features = [c for c in df.columns if c != target]

n = len(num_features)
cols = 3
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
axes = axes.flatten()

for i, feat in enumerate(num_features):
    ax = axes[i]
    x = df[feat]
    y = df[target]
    
    # Scatter
    ax.scatter(x, y, alpha=0.08, s=3, color='steelblue')
    
    # Droite de regression
    coefs = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, np.polyval(coefs, x_line), 'r-', lw=2)
    
    # Coefficient de correlation
    r = np.corrcoef(x, y)[0, 1]
    ax.set_title(f'{feat}  (r = {r:.3f})', fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel('median_house_value ($)')

# Masquer les axes vides
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Linearite : chaque feature vs prix median', fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 1. Analyse de colinearite

On identifie les paires de features avec $|r| > 0.8$ (fortement correlees).  
Pour chaque paire, on supprime la feature la moins correlee avec la cible.

In [ ]:
target = 'median_house_value'
features = [c for c in df.columns if c != target]

corr = df[features].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Correlation entre features')
plt.tight_layout()
plt.show()

# Paires fortement correlees
threshold = 0.8
pairs = []
for i in range(len(corr)):
    for j in range(i + 1, len(corr)):
        if abs(corr.iloc[i, j]) > threshold:
            pairs.append((corr.index[i], corr.columns[j], corr.iloc[i, j]))

print(f"\nPaires avec |r| > {threshold} :")
for a, b, r in pairs:
    print(f"  {a} <-> {b} : r = {r:.3f}")

In [ ]:
# Suppression des features redondantes
target_corr = df[features].corrwith(df[target]).abs()

to_drop = set()
for a, b, r in pairs:
    drop = a if target_corr[a] < target_corr[b] else b
    to_drop.add(drop)
    print(f"  {a} (|r_target|={target_corr[a]:.3f}) vs {b} (|r_target|={target_corr[b]:.3f}) -> drop {drop}")

features_clean = [f for f in features if f not in to_drop]
print(f"\nFeatures conservees ({len(features_clean)}) : {features_clean}")
print(f"Features supprimees ({len(to_drop)}) : {to_drop}")

## 2. Train / Test split (70/30)

In [ ]:
X = df[features_clean].values
y = df[target].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Ajout de la constante (intercept) pour statsmodels
X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test)

print(f"Train : {X_train.shape[0]} echantillons ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test  : {X_test.shape[0]} echantillons ({X_test.shape[0]/len(X)*100:.0f}%)")

## 3. Regression lineaire (statsmodels OLS)

Le modele apprend les coefficients $\theta$ par la methode des **moindres carres ordinaires** (OLS) :

$$\hat{y} = X \theta \quad \text{avec} \quad \theta = (X^T X)^{-1} X^T y$$

`statsmodels` fournit un **resume statistique complet** : coefficients, p-values, intervalles de confiance, R², F-statistic, etc.

In [ ]:
model = sm.OLS(y_train, X_train_sm).fit()

# Resume statistique complet
print(model.summary(xname=['const'] + features_clean))

## 4. Predictions

In [ ]:
y_pred_train = model.predict(X_train_sm)
y_pred_test = model.predict(X_test_sm)

# Scatter plot : predit vs reel
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, y_true, y_pred) in zip(axes, [('Train (70%)', y_train, y_pred_train), ('Test (30%)', y_test, y_pred_test)]):
    ax.scatter(y_true, y_pred, alpha=0.1, s=5)
    ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=1.5, label='parfait')
    ax.set_xlabel('Prix reel ($)')
    ax.set_ylabel('Prix predit ($)')
    ax.set_title(f'{name}')
    ax.legend()
plt.suptitle('Regression lineaire (statsmodels OLS) — Predit vs Reel', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Evaluation de la precision — Moyenne et ecart-type des erreurs (en %)

$$e_{\%,i} = \frac{y_i - \hat{y}_i}{y_i} \times 100$$

| Metrique | Formule | Interpretation |
|----------|---------|----------------|
| **Moyenne des erreurs (%)** | $\bar{e}_\% = \frac{1}{n}\sum \frac{e_i}{y_i} \times 100$ | Biais relatif du modele (0% = pas de biais systematique) |
| **Ecart-type des erreurs (%)** | $\sigma_{e\%} = \text{std}\left(\frac{e_i}{y_i} \times 100\right)$ | Dispersion relative des erreurs (plus bas = plus precis) |

In [ ]:
def evaluer_erreurs(y_true, y_pred, nom):
    """Calcule et affiche la moyenne et l'ecart-type des erreurs en pourcentage."""
    erreurs_pct = ((y_true - y_pred) / y_true) * 100
    
    moyenne = np.mean(erreurs_pct)
    ecart_type = np.std(erreurs_pct)
    
    print(f"--- {nom} ---")
    print(f"  Moyenne des erreurs (biais)      : {moyenne:>+8.2f} %")
    print(f"  Ecart-type des erreurs (spread)   : {ecart_type:>8.2f} %")
    print(f"  Intervalle 68% (1 sigma)          : [{moyenne - ecart_type:>+.2f}%, {moyenne + ecart_type:>+.2f}%]")
    print(f"  Intervalle 95% (2 sigma)          : [{moyenne - 2*ecart_type:>+.2f}%, {moyenne + 2*ecart_type:>+.2f}%]")
    print()
    
    return erreurs_pct, moyenne, ecart_type

err_train, mu_train, std_train = evaluer_erreurs(y_train, y_pred_train, 'Train (70%)')
err_test, mu_test, std_test = evaluer_erreurs(y_test, y_pred_test, 'Test (30%)')

In [ ]:
# Distribution des erreurs relatives (%)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (nom, erreurs, mu, sigma) in zip(axes, [
    ('Train (70%)', err_train, mu_train, std_train),
    ('Test (30%)', err_test, mu_test, std_test)
]):
    ax.hist(erreurs, bins=60, density=True, alpha=0.7, color='steelblue', edgecolor='white')
    
    # Courbe gaussienne theorique
    x = np.linspace(erreurs.min(), erreurs.max(), 300)
    gauss = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)
    ax.plot(x, gauss, 'r-', lw=2, label=f'Gaussienne\n$\\mu$={mu:.2f}%, $\\sigma$={sigma:.2f}%')
    
    # Lignes verticales pour mu et sigma
    ax.axvline(mu, color='red', linestyle='--', lw=1, alpha=0.8, label=f'Moyenne = {mu:.2f}%')
    ax.axvline(mu - sigma, color='orange', linestyle=':', lw=1, alpha=0.8)
    ax.axvline(mu + sigma, color='orange', linestyle=':', lw=1, alpha=0.8, label=f'$\\pm 1\\sigma$ = {sigma:.2f}%')
    
    ax.set_xlabel('Erreur relative (%)')
    ax.set_ylabel('Densite')
    ax.set_title(f'{nom}')
    ax.legend(fontsize=8)

plt.suptitle('Distribution des erreurs relatives (%)', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Resume des resultats

$$R^2 = 1 - \frac{V(\text{reel})}{V(\text{theorique})} = 1 - \frac{\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{n} (y_i - \bar{y})^2}$$

| Variable | Description |
|----------|-------------|
| $V(\text{reel}) = \sum(y_i - \hat{y}_i)^2$ | Variance residuelle : erreur reelle du modele |
| $V(\text{theorique}) = \sum(y_i - \bar{y})^2$ | Variance totale theorique : dispersion totale des donnees autour de la moyenne |

- $R^2 = 1$ : prediction parfaite
- $R^2 = 0$ : le modele ne fait pas mieux que predire la moyenne $\bar{y}$
- $R^2 < 0$ : le modele fait pire que la moyenne

In [ ]:
resume = pd.DataFrame({
    'Train': {
        'Moyenne erreurs (%)': mu_train,
        'Ecart-type erreurs (%)': std_train,
        'MAE ($)': mean_absolute_error(y_train, y_pred_train),
        'RMSE ($)': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        'R2': r2_score(y_train, y_pred_train),
    },
    'Test': {
        'Moyenne erreurs (%)': mu_test,
        'Ecart-type erreurs (%)': std_test,
        'MAE ($)': mean_absolute_error(y_test, y_pred_test),
        'RMSE ($)': np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'R2': r2_score(y_test, y_pred_test),
    }
})

print(resume.to_string(float_format=lambda x: f'{x:.2f}'))

print(f"\n--- Conclusion ---")
print(f"Le modele de regression lineaire a un ecart-type d'erreur de {std_test:.2f}% sur le jeu de test.")
print(f"Cela signifie que 68% des predictions sont a +/- {std_test:.2f}% du prix reel,")
print(f"et 95% des predictions sont a +/- {2*std_test:.2f}% du prix reel.")

In [ ]:
# Graphique de synthese : erreur moyenne +/- ecart-type (%) pour train et test
fig, ax = plt.subplots(figsize=(6, 4))

labels = ['Train', 'Test']
moyennes = [mu_train, mu_test]
ecarts = [std_train, std_test]

x_pos = range(len(labels))
bars = ax.bar(x_pos, moyennes, yerr=ecarts, capsize=10, color=['steelblue', 'coral'], 
              edgecolor='white', alpha=0.8, width=0.5)

ax.axhline(0, color='black', linestyle='-', lw=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(labels)
ax.set_ylabel('Erreur relative (%)')
ax.set_title('Moyenne $\\pm$ ecart-type des erreurs (%)', fontweight='bold')

for i, (m, s) in enumerate(zip(moyennes, ecarts)):
    ax.text(i, m + s + 1, f'$\\mu$={m:.2f}%\n$\\sigma$={s:.2f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

---
# PARTIE 3 — Random Forest

**Modele** : Random Forest Regressor (ensemble d'arbres de decision)

$$\hat{y} = \frac{1}{B} \sum_{b=1}^{B} T_b(x)$$

ou $B$ est le nombre d'arbres et $T_b(x)$ la prediction de l'arbre $b$.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

df = pd.read_csv('./datasets/amputed_normalized_housing_data.csv')
print(f"Shape : {df.shape}")
df.head()

## 1. Analyse de colinearite

On identifie les paires de features avec $|r| > 0.8$ (fortement correlees).  
Pour chaque paire, on supprime la feature la moins correlee avec la cible.

In [ ]:
target = 'median_house_value'
features = [c for c in df.columns if c != target]

corr = df[features].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Correlation entre features')
plt.tight_layout()
plt.show()

# Paires fortement correlees
threshold = 0.8
pairs = []
for i in range(len(corr)):
    for j in range(i + 1, len(corr)):
        if abs(corr.iloc[i, j]) > threshold:
            pairs.append((corr.index[i], corr.columns[j], corr.iloc[i, j]))

print(f"\nPaires avec |r| > {threshold} :")
for a, b, r in pairs:
    print(f"  {a} <-> {b} : r = {r:.3f}")

In [ ]:
# Suppression des features redondantes
target_corr = df[features].corrwith(df[target]).abs()

to_drop = set()
for a, b, r in pairs:
    drop = a if target_corr[a] < target_corr[b] else b
    to_drop.add(drop)
    print(f"  {a} (|r_target|={target_corr[a]:.3f}) vs {b} (|r_target|={target_corr[b]:.3f}) -> drop {drop}")

features_clean = [f for f in features if f not in to_drop]
print(f"\nFeatures conservees ({len(features_clean)}) : {features_clean}")
print(f"Features supprimees ({len(to_drop)}) : {to_drop}")

### Dendrogramme — Clustering hierarchique des features

Le dendrogramme montre la **proximite** entre les features basee sur leur correlation. La distance utilisee est $1 - |r|$.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

# Matrice de distance basee sur la correlation : d = 1 - |r|
corr_matrix = df[features].corr().abs()
distance_matrix = 1 - corr_matrix

# Conversion en forme condensee pour scipy
dist_condensed = squareform(distance_matrix, checks=False)

# Clustering hierarchique (methode de Ward)
linked = linkage(dist_condensed, method='ward')

fig, ax = plt.subplots(figsize=(12, 6))
dendro = dendrogram(
    linked,
    labels=features,
    ax=ax,
    leaf_rotation=45,
    leaf_font_size=10,
    color_threshold=0.7,
    above_threshold_color='gray',
)

ax.set_ylabel('Distance (1 - |r|)', fontsize=11)
ax.set_title('Dendrogramme — Clustering hierarchique des features', fontweight='bold', fontsize=13)
ax.axhline(y=0.7, color='red', linestyle='--', lw=1, alpha=0.7, label='Seuil colinearite (|r| > 0.8 => d < 0.2)')

plt.tight_layout()
plt.show()

print("Lecture : les features qui fusionnent a faible distance sont fortement correlees.")
print("Les groupes identiques a l'analyse de colinearite apparaissent clairement.")

## 2. Train / Test split (70/30)

In [ ]:
X = df[features_clean].values
y = df[target].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"Train : {X_train.shape[0]} echantillons ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test  : {X_test.shape[0]} echantillons ({X_test.shape[0]/len(X)*100:.0f}%)")

## 3. Random Forest Regressor

Architecture : **5 features → 100 arbres (bootstrap) → moyenne → prix**

**Avantages** par rapport a la regression lineaire :
- Capture les relations **non lineaires**
- Robuste aux outliers et au bruit

In [ ]:
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

print(f"Nombre d'arbres       : {model.n_estimators}")
print(f"Profondeur max        : {model.max_depth or 'illimitee'}")
print(f"Features par split    : {model.max_features}")
print(f"Nombre de features    : {model.n_features_in_}")

### Architecture du Random Forest

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(16, 10))
ax.axis('off')
ax.set_xlim(-1, 17)
ax.set_ylim(-1, 11)

# --- Input block ---
input_box = FancyBboxPatch((0, 4.2), 3, 2.6, boxstyle="round,pad=0.2", fc='#e8f0fe', ec='steelblue', lw=2)
ax.add_patch(input_box)
ax.text(1.5, 6.4, 'Entree (5 features)', ha='center', fontweight='bold', fontsize=9)
for j, feat in enumerate(features_clean):
    ax.text(1.5, 5.9 - j * 0.4, feat, ha='center', fontsize=7, fontstyle='italic')

# --- Tree blocks ---
n_trees_show = 5
tree_x = 5.5
tree_spacing = 1.8
tree_y_start = 9.0

for t in range(n_trees_show):
    y = tree_y_start - t * tree_spacing
    box = FancyBboxPatch((tree_x, y - 0.5), 3.5, 1.0, boxstyle="round,pad=0.15",
                          fc='#d4edda', ec='forestgreen', lw=1.5)
    ax.add_patch(box)
    if t < n_trees_show - 1:
        ax.text(tree_x + 1.75, y, f'Arbre {t+1}', ha='center', va='center', fontweight='bold', fontsize=9)
    else:
        ax.text(tree_x + 1.75, y, f'Arbre {model.n_estimators}', ha='center', va='center', fontweight='bold', fontsize=9)
    
    # Arrow from input to tree
    ax.annotate('', xy=(tree_x, y), xytext=(3, 5.5),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1, alpha=0.5))

# Dots between tree 4 and tree 100
dots_y = tree_y_start - 3 * tree_spacing
ax.text(tree_x + 1.75, dots_y, '...', ha='center', va='center', fontsize=20, color='gray')

# --- Moyenne block ---
avg_x = 11.5
avg_box = FancyBboxPatch((avg_x, 4.0), 2.5, 3.0, boxstyle="round,pad=0.2", fc='#fff3cd', ec='orange', lw=2)
ax.add_patch(avg_box)
ax.text(avg_x + 1.25, 6.5, 'MOYENNE', ha='center', fontweight='bold', fontsize=10)
ax.text(avg_x + 1.25, 5.8, r'$\hat{y} = \frac{1}{B}\sum T_b(x)$', ha='center', fontsize=10)
ax.text(avg_x + 1.25, 5.1, f'B = {model.n_estimators} arbres', ha='center', fontsize=8, color='gray')

# Arrows from trees to average
for t in range(n_trees_show):
    y = tree_y_start - t * tree_spacing
    if t != 3:  # skip dots row
        ax.annotate('', xy=(avg_x, 5.5), xytext=(tree_x + 3.5, y),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1, alpha=0.5))

# --- Output block ---
out_x = 15
out_box = FancyBboxPatch((out_x, 4.5), 1.8, 2.0, boxstyle="round,pad=0.2", fc='#f8d7da', ec='#dc3545', lw=2)
ax.add_patch(out_box)
ax.text(out_x + 0.9, 6.0, 'Sortie', ha='center', fontweight='bold', fontsize=10)
ax.text(out_x + 0.9, 5.3, r'$\hat{y}$', ha='center', fontsize=14)
ax.text(out_x + 0.9, 4.8, 'Prix ($)', ha='center', fontsize=8, color='gray')

# Arrow from average to output
ax.annotate('', xy=(out_x, 5.5), xytext=(avg_x + 2.5, 5.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

plt.title('Architecture Random Forest : 5 features -> 100 arbres -> moyenne -> prix', fontweight='bold', fontsize=12, pad=20)
plt.tight_layout()
plt.show()

### Visualisation d'un arbre individuel

Arbre n°0 de la foret, limite a une profondeur de 3 pour la lisibilite.

In [ ]:
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(
    model.estimators_[0],
    max_depth=3,
    feature_names=features_clean,
    filled=True,
    rounded=True,
    fontsize=8,
    ax=ax,
    impurity=False,
)
ax.set_title('Arbre n°0 de la foret (profondeur limitee a 3)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# Stats sur la profondeur des arbres
depths = [est.get_depth() for est in model.estimators_]
print(f"Profondeur des {len(depths)} arbres :")
print(f"  Min : {min(depths)}, Max : {max(depths)}, Moyenne : {np.mean(depths):.1f}")

## 4. Importance des features

Le Random Forest mesure l'importance de chaque feature par la **reduction moyenne de l'impurete** (MDI).

In [ ]:
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

print("Importance des features :")
for i in indices:
    print(f"  {features_clean[i]:25s} : {importances[i]:.4f}")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh([features_clean[i] for i in indices[::-1]], importances[indices[::-1]], color='steelblue', edgecolor='white')
ax.set_xlabel('Importance (MDI)')
ax.set_title('Importance des features — Random Forest', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Predictions

In [ ]:
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Scatter plot : predit vs reel
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, y_true, y_pred) in zip(axes, [('Train (70%)', y_train, y_pred_train), ('Test (30%)', y_test, y_pred_test)]):
    ax.scatter(y_true, y_pred, alpha=0.1, s=5)
    ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=1.5, label='parfait')
    ax.set_xlabel('Prix reel ($)')
    ax.set_ylabel('Prix predit ($)')
    ax.set_title(f'{name}')
    ax.legend()
plt.suptitle('Random Forest — Predit vs Reel', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Evaluation — Moyenne et ecart-type des erreurs (%)

In [ ]:
def evaluer_erreurs_rf(y_true, y_pred, nom):
    """Calcule et affiche la moyenne et l'ecart-type des erreurs en pourcentage."""
    erreurs_pct = ((y_true - y_pred) / y_true) * 100
    
    moyenne = np.mean(erreurs_pct)
    ecart_type = np.std(erreurs_pct)
    
    print(f"--- {nom} ---")
    print(f"  Moyenne des erreurs (biais)      : {moyenne:>+8.2f} %")
    print(f"  Ecart-type des erreurs (spread)   : {ecart_type:>8.2f} %")
    print(f"  Intervalle 68% (1 sigma)          : [{moyenne - ecart_type:>+.2f}%, {moyenne + ecart_type:>+.2f}%]")
    print(f"  Intervalle 95% (2 sigma)          : [{moyenne - 2*ecart_type:>+.2f}%, {moyenne + 2*ecart_type:>+.2f}%]")
    print()
    
    return erreurs_pct, moyenne, ecart_type

err_train, mu_train, std_train = evaluer_erreurs_rf(y_train, y_pred_train, 'Train (70%)')
err_test, mu_test, std_test = evaluer_erreurs_rf(y_test, y_pred_test, 'Test (30%)')

In [ ]:
# Distribution des erreurs relatives (%)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (nom, erreurs, mu, sigma) in zip(axes, [
    ('Train (70%)', err_train, mu_train, std_train),
    ('Test (30%)', err_test, mu_test, std_test)
]):
    ax.hist(erreurs, bins=60, density=True, alpha=0.7, color='forestgreen', edgecolor='white')
    
    # Courbe gaussienne theorique
    x = np.linspace(erreurs.min(), erreurs.max(), 300)
    gauss = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)
    ax.plot(x, gauss, 'r-', lw=2, label=f'Gaussienne\n$\\mu$={mu:.2f}%, $\\sigma$={sigma:.2f}%')
    
    # Lignes verticales pour mu et sigma
    ax.axvline(mu, color='red', linestyle='--', lw=1, alpha=0.8, label=f'Moyenne = {mu:.2f}%')
    ax.axvline(mu - sigma, color='orange', linestyle=':', lw=1, alpha=0.8)
    ax.axvline(mu + sigma, color='orange', linestyle=':', lw=1, alpha=0.8, label=f'$\\pm 1\\sigma$ = {sigma:.2f}%')
    
    ax.set_xlabel('Erreur relative (%)')
    ax.set_ylabel('Densite')
    ax.set_title(f'{nom}')
    ax.legend(fontsize=8)

plt.suptitle('Distribution des erreurs relatives (%) — Random Forest', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Resume des resultats — Random Forest

In [ ]:
resume = pd.DataFrame({
    'Train': {
        'Moyenne erreurs (%)': mu_train,
        'Ecart-type erreurs (%)': std_train,
        'MAE ($)': mean_absolute_error(y_train, y_pred_train),
        'RMSE ($)': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        'R2': r2_score(y_train, y_pred_train),
    },
    'Test': {
        'Moyenne erreurs (%)': mu_test,
        'Ecart-type erreurs (%)': std_test,
        'MAE ($)': mean_absolute_error(y_test, y_pred_test),
        'RMSE ($)': np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'R2': r2_score(y_test, y_pred_test),
    }
})

print(resume.to_string(float_format=lambda x: f'{x:.2f}'))

print(f"\n--- Conclusion ---")
print(f"Le Random Forest a un ecart-type d'erreur de {std_test:.2f}% sur le jeu de test.")
print(f"68% des predictions sont a +/- {std_test:.2f}% du prix reel,")
print(f"et 95% des predictions sont a +/- {2*std_test:.2f}% du prix reel.")

In [ ]:
# Graphique de synthese : erreur moyenne +/- ecart-type (%) pour train et test
fig, ax = plt.subplots(figsize=(6, 4))

labels = ['Train', 'Test']
moyennes = [mu_train, mu_test]
ecarts = [std_train, std_test]

x_pos = range(len(labels))
bars = ax.bar(x_pos, moyennes, yerr=ecarts, capsize=10, edgecolor='white', alpha=0.8, width=0.5)

ax.axhline(0, color='black', linestyle='-', lw=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(labels)
ax.set_ylabel('Erreur relative (%)')
ax.set_title('Moyenne $\\pm$ ecart-type des erreurs (%) — Random Forest', fontweight='bold')

for i, (m, s) in enumerate(zip(moyennes, ecarts)):
    ax.text(i, m + s + 1, f'$\\mu$={m:.2f}%\n$\\sigma$={s:.2f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

---
# PARTIE 4 — Perceptron Multi-Couches (MLP)

**Modele** : Multi-Layer Perceptron (regression)

$$h^{(l)} = \text{ReLU}(W^{(l)} h^{(l-1)} + b^{(l)}), \quad \hat{y} = W^{(out)} h^{(L)} + b^{(out)}$$

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

df = pd.read_csv('./datasets/amputed_normalized_housing_data.csv')
print(f"Shape : {df.shape}")
df.head()

## 1. Analyse de colinearite

Seuil : $|r| > 0.8$

In [ ]:
target = 'median_house_value'
features = [c for c in df.columns if c != target]

# Matrice de corrélation entre features
corr = df[features].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Corrélation entre features')
plt.tight_layout()
plt.show()

# Paires fortement corrélées (|r| > 0.8)
threshold = 0.8
pairs = []
for i in range(len(corr)):
    for j in range(i + 1, len(corr)):
        if abs(corr.iloc[i, j]) > threshold:
            pairs.append((corr.index[i], corr.columns[j], corr.iloc[i, j]))

print(f"\nPaires avec |r| > {threshold} :")
for a, b, r in pairs:
    print(f"  {a} <-> {b} : r = {r:.3f}")

In [ ]:
# Pour chaque paire colinéaire, on garde celle qui corrèle le mieux avec la cible
target_corr = df[features].corrwith(df[target]).abs()

to_drop = set()
for a, b, r in pairs:
    drop = a if target_corr[a] < target_corr[b] else b
    to_drop.add(drop)
    print(f"  {a} (|r_target|={target_corr[a]:.3f}) vs {b} (|r_target|={target_corr[b]:.3f}) -> drop {drop}")

features_clean = [f for f in features if f not in to_drop]
print(f"\nFeatures conservées ({len(features_clean)}) : {features_clean}")
print(f"Features supprimées ({len(to_drop)}) : {to_drop}")

## 2. Train / Test split (70/30)

In [ ]:
X = df[features_clean].values
y = df[target].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"Train : {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test  : {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)")

## 3. Perceptron Multi-Couches (MLPRegressor)

Architecture : **5 → 64 → 32 → 1** (2 couches cachees, activation ReLU, optimiseur Adam)

$$h_1 = \text{ReLU}(W_1 x + b_1) \quad h_2 = \text{ReLU}(W_2 h_1 + b_2) \quad \hat{y} = W_3 h_2 + b_3$$

$$\mathcal{L} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

In [ ]:
model = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42,
)

model.fit(X_train, y_train)
print(f"Convergé en {model.n_iter_} epochs")
print(f"Architecture : {[X_train.shape[1]]} → {list(model.hidden_layer_sizes)} → [1]")
print(f"Nombre de poids : {sum(c.size for c in model.coefs_) + sum(b.size for b in model.intercepts_)}")

# Courbe de loss
plt.figure(figsize=(8, 4))
plt.plot(model.loss_curve_, label='train loss')
if model.validation_scores_:
    plt.plot(model.validation_scores_, label='validation R²', alpha=0.7)
plt.xlabel('Epoch')
plt.ylabel('Loss / Score')
plt.title('Courbe d\'apprentissage')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Evaluation

| Metrique | Formule | Echelle |
|----------|---------|---------|
| **MAE** | $\frac{1}{n}\sum\|y_i - \hat{y}_i\|$ | Dollars ($) |
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ | Dollars ($) |
| **MAPE** | $\frac{100}{n}\sum\left\|\frac{y_i - \hat{y}_i}{y_i}\right\|$ | Pourcentage (%) |
| **R²** | $1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$ | Sans unite [0,1] |

In [ ]:
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

metrics = {}
for name, y_true, y_pred in [('Train', y_train, y_pred_train), ('Test', y_test, y_pred_test)]:
    metrics[name] = {
        'MAE ($)': mean_absolute_error(y_true, y_pred),
        'RMSE ($)': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAPE (%)': mean_absolute_percentage_error(y_true, y_pred) * 100,
        'R²': r2_score(y_true, y_pred),
    }

metrics_df = pd.DataFrame(metrics).T
print(metrics_df.to_string(float_format=lambda x: f'{x:.2f}'))

# Scatter plot prédit vs réel
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, y_true, y_pred) in zip(axes, [('Train (70%)', y_train, y_pred_train), ('Test (30%)', y_test, y_pred_test)]):
    ax.scatter(y_true, y_pred, alpha=0.1, s=5)
    ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=1.5, label='parfait')
    ax.set_xlabel('Prix réel ($)')
    ax.set_ylabel('Prix prédit ($)')
    ax.set_title(f'{name}')
    ax.legend()
plt.suptitle('MLP — Prédit vs Réel', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Architecture du MLP

Visualisation du reseau **5 → 64 → 32 → 1**.

In [ ]:
import matplotlib.patches as mpatches

layer_sizes = [len(features_clean), 64, 32, 1]
layer_labels = ['Entrée', 'Couche cachée 1\nReLU (64)', 'Couche cachée 2\nReLU (32)', 'Sortie\nIdentité']
layer_colors = ['#e8f0fe', '#fff3cd', '#fff3cd', '#d4edda']
layer_edge = ['steelblue', 'orange', 'orange', 'green']
max_display = [layer_sizes[0], 8, 6, 1]  # cap displayed neurons for readability

fig, ax = plt.subplots(figsize=(16, 8))
ax.axis('off')
ax.set_xlim(-0.5, len(layer_sizes) * 2.5)
ax.set_ylim(-1, max(max_display) + 2)
ax.set_title('Architecture MLP : 5 → 64 (ReLU) → 32 (ReLU) → 1', fontsize=14, fontweight='bold', pad=20)

node_positions = []

for l, (size, display, label, fc, ec) in enumerate(zip(layer_sizes, max_display, layer_labels, layer_colors, layer_edge)):
    x = l * 2.5 + 0.5
    positions = []
    offset = (max(max_display) - display) / 2

    for i in range(display):
        y = offset + display - i
        circle = plt.Circle((x, y), 0.25, fc=fc, ec=ec, lw=1.5, zorder=5)
        ax.add_patch(circle)
        positions.append((x, y))

        # Label input neurons with feature names
        if l == 0:
            ax.text(x, y, features_clean[i], ha='center', va='center', fontsize=6, zorder=6)
        elif l == len(layer_sizes) - 1:
            ax.text(x, y, 'ŷ', ha='center', va='center', fontsize=11, fontweight='bold', zorder=6)

    # Show "..." if truncated
    if size > display:
        y_dots = offset + 0.3
        ax.text(x, y_dots, f'⋮\n({size} neurones)', ha='center', va='center', fontsize=8, color='gray')

    # Layer label
    ax.text(x, -0.5, label, ha='center', va='top', fontsize=9, fontstyle='italic', color='gray')

    node_positions.append(positions)

# Draw connections between layers (subsample for readability)
for l in range(len(node_positions) - 1):
    src = node_positions[l]
    dst = node_positions[l + 1]
    for sx, sy in src:
        for dx, dy in dst:
            ax.plot([sx + 0.25, dx - 0.25], [sy, dy], color='gray', alpha=0.12, lw=0.5, zorder=1)

# Sample prediction
sample = df[features_clean].iloc[0].values
sample_pred = model.predict(sample.reshape(1, -1))[0]
sample_target = df[target].iloc[0]
out_x, out_y = node_positions[-1][0]
ax.text(out_x + 0.5, out_y + 0.2, f'prédit : {sample_pred:,.0f} $', ha='left', fontsize=9, color='green')
ax.text(out_x + 0.5, out_y - 0.2, f'réel   : {sample_target:,.0f} $', ha='left', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

## 6. Export du modele

In [ ]:
import joblib

joblib.dump(model, './models/mlp_model.joblib')
joblib.dump(features_clean, './models/model_features.joblib')
print("Modèle exporté → ./models/mlp_model.joblib")
print(f"Features utilisées : {features_clean}")

---
# PARTIE 5 — Industrialisation (predict.py)

Script de prediction en ligne de commande. Charge le modele MLP sauvegarde et le scaler pour predire a partir de donnees brutes saisies par l'utilisateur.

```python
"""
Prédiction du prix d'un appartement avec le MLP entraîné.
Saisie manuelle des paramètres → prix estimé.

Usage : python predict.py
"""

import numpy as np
import joblib

# Charger le modèle et le scaler
model = joblib.load('./models/mlp_model.joblib')
scaler = joblib.load('./models/scaler.joblib')
feature_cols = joblib.load('./models/feature_cols.joblib')
model_features = joblib.load('./models/model_features.joblib')

OCEAN_CATEGORIES = {
    '0': ('<1H OCEAN', 0),
    '1': ('INLAND', 1),
    '2': ('ISLAND', 2),
    '3': ('NEAR BAY', 3),
    '4': ('NEAR OCEAN', 4),
}


def get_input():
    print("=" * 55)
    print("  PREDICTION DU PRIX IMMOBILIER (MLP Perceptron)")
    print("=" * 55)
    print()

    raw = {}

    raw['longitude'] = float(input("Longitude (ex: -122.23) : "))
    raw['latitude'] = float(input("Latitude (ex: 37.88) : "))
    raw['housing_median_age'] = float(input("Age médian du logement (ex: 41) : "))
    raw['total_rooms'] = float(input("Nombre total de pièces du bloc (ex: 880) : "))
    raw['total_bedrooms'] = float(input("Nombre total de chambres du bloc (ex: 129) : "))
    raw['population'] = float(input("Population du bloc (ex: 322) : "))
    raw['households'] = float(input("Nombre de ménages du bloc (ex: 126) : "))
    raw['median_income'] = float(input("Revenu médian (en dizaines de k$, ex: 8.3) : "))

    print()
    print("Proximité océan :")
    for k, (name, _) in OCEAN_CATEGORIES.items():
        print(f"  {k} = {name}")
    choice = input("Choix (0-4) : ")
    raw['ocean_proximity'] = OCEAN_CATEGORIES[choice][1]

    return raw


def predict(raw):
    # Construire le vecteur dans l'ordre du scaler (feature_cols)
    values = np.array([[raw[col] for col in feature_cols]])

    # Normaliser avec le scaler (même que l'entraînement)
    values_scaled = scaler.transform(values)

    # Sélectionner les features utilisées par le modèle
    feature_indices = [feature_cols.index(f) for f in model_features]
    values_model = values_scaled[:, feature_indices]

    # Prédire
    price = model.predict(values_model)[0]
    return price


def main():
    try:
        raw = get_input()
        price = predict(raw)

        print()
        print("-" * 55)
        print(f"  Prix estimé : {price:,.0f} $")
        print("-" * 55)
        print()
        print("  (MAPE ~28% → marge d'erreur estimée :")
        lower = price * 0.72
        upper = price * 1.28
        print(f"   entre {lower:,.0f} $ et {upper:,.0f} $)")
        print()

    except KeyboardInterrupt:
        print("\nAnnulé.")
    except Exception as e:
        print(f"\nErreur : {e}")


if __name__ == '__main__':
    main()
```